# FPSA-Prime portable verification notebook

This notebook checks the corrected implementation before benchmark accuracy is interpreted. It validates parameter matching, corrected GMRES convergence, stability-estimator calibration, fixed-unroll train/eval identity, and an optional tiny implicit-training smoke test.

The legacy v0 GMRES curve is shown only as a **diagnostic**. Its exact degradation is device and linear-algebra-library dependent, so failing to reproduce a particular >100× legacy spike is not a correctness failure. The corrected solver must reach tolerance, stop early, and preserve its converged solution as its maximum budget grows.


In [ ]:
# Colab controls
REPO_URL = "https://github.com/mrinal18/fpsa.git"
BRANCH = "fpsa-prime-v0"
QUICK = True
RUN_TINY_TRAINING = True
RUN_FULL_BENCHMARK = False
FULL_SEEDS = [0]  # expand to [0, 1, 2] after one clean GPU run


## 1. Checkout the latest branch and install dependencies


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
target = Path("/content/fpsa") if IN_COLAB else Path.cwd()

if not (target / ".git").exists():
    if target.exists() and target != Path.cwd():
        subprocess.run(["rm", "-rf", str(target)], check=True)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(target)], check=True)
else:
    os.chdir(target)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)

os.chdir(target)
sys.path.insert(0, str(target))
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pytest", "pandas", "matplotlib"],
        check=True,
    )

import pandas as pd
import matplotlib.pyplot as plt
import torch

print({
    "git_sha": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})


## 2. Run the portable numerical verification harness


In [ ]:
from experiments.fpsa_prime.verification_portable import run_verification

summary = run_verification(
    output_dir="results/verification_colab",
    device="auto",
    quick=QUICK,
    run_tests=True,
    run_training=RUN_TINY_TRAINING,
)

print(json.dumps({
    "tests": summary["tests"].splitlines()[-1],
    "parameter_gap_percent": 100 * summary["hero_block_parameter_gap_fraction"],
    "fixed_unroll": summary["fixed_unroll"],
    "gmres_diagnostic": summary["gmres_diagnostic"],
    "plots": summary["plots"],
}, indent=2))


## 3. Inspect tables


In [ ]:
parameter_table = pd.DataFrame(summary["parameter_counts"])
gmres_table = pd.DataFrame(summary["gmres"])
stability_table = pd.DataFrame(summary["stability_power_sweep"])

display(parameter_table)
display(gmres_table)
display(stability_table)
if summary["tiny_training"] is not None:
    display(pd.DataFrame(summary["tiny_training"]))


## 4. Inspect plots


In [ ]:
from IPython.display import Image, display

for plot_name in summary["plots"]:
    print(plot_name)
    display(Image(filename=f"results/verification_colab/{plot_name}"))


## 5. Optional full controlled Maze experiment

This is the first section that can support an architecture-level comparison. It trains FPSA-Prime and the block-DEQ control through the same launcher, data, optimizer, schedule, seed handling, evaluation, and JSON schema.


In [ ]:
if RUN_FULL_BENCHMARK:
    output_dir = Path("results/controlled_colab")
    output_dir.mkdir(parents=True, exist_ok=True)

    for seed in FULL_SEEDS:
        shared = [
            "--task", "maze",
            "--maze_size", "7",
            "--extra_sizes", "9", "11",
            "--device", "cuda" if torch.cuda.is_available() else "cpu",
            "--seed", str(seed),
            "--steps", "1200",
            "--batch_size", "32",
            "--lr", "3e-3",
            "--hidden", "128",
            "--heads", "4",
            "--max_iter", "16",
        ]
        prime_cmd = [
            sys.executable, "-m", "experiments.fpsa_prime.controlled_compare",
            "--family", "prime", "--arch", "fpsa_prime",
            *shared,
            "--stability_power_steps", "4",
            "--output", str(output_dir / f"prime_maze7_s{seed}.json"),
        ]
        block_cmd = [
            sys.executable, "-m", "experiments.fpsa_prime.controlled_compare",
            "--family", "block", "--arch", "deq_block",
            *shared,
            "--output", str(output_dir / f"deq_maze7_s{seed}.json"),
        ]
        subprocess.run(prime_cmd, check=True)
        subprocess.run(block_cmd, check=True)

    rows = []
    for path in sorted(output_dir.glob("*.json")):
        result = json.loads(path.read_text())
        rows.append({
            "family": result["family"],
            "arch": result["arch"],
            "seed": result["seed"],
            "params": result["params"],
            "maze7_em": result["final"]["exact_match"],
            "maze9_em": result["extra"]["size9"]["exact_match"],
            "maze11_em": result["extra"]["size11"]["exact_match"],
            "forward_nfe": result["final"]["mean_function_evals"],
            "elapsed_seconds": result["elapsed_seconds"],
        })
    full_results = pd.DataFrame(rows)
    display(full_results)
    display(full_results.groupby(["family", "arch"]).agg(["mean", "std"]))
else:
    print("Full benchmark disabled. Set RUN_FULL_BENCHMARK=True after the quick checks pass.")


## Pass criteria

- all repository correctness tests pass;
- hero/block parameter gap is below 1%;
- corrected GMRES reaches `1e-5` on the actual Maze-shaped FPSA VJP;
- corrected GMRES stops early and does not lose the solution when the maximum budget increases;
- legacy GMRES numbers are recorded only as a runtime-dependent diagnostic;
- fixed-unroll logits and residuals are identical in train and eval;
- the known linear stability map estimates `sigma=0.7` with a positive corrective gradient;
- the optional tiny training run keeps forward residual below `1e-4`, adjoint residual below `1e-5`, and all values finite.
